
# Transfer Learning in PyTorch: ResNet vs MobileNet

In this live coding session, we will:

- Understand transfer learning
- Compare ResNet50 and MobileNetV3
- Implement feature extraction (freeze backbone)
- Implement fine-tuning
- Compare performance
- Analyze domain shift

Framework: **PyTorch**


In [ ]:

# Core libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torchvision import models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



## Dataset

We use CIFAR-10 resized to 224x224 to match ImageNet pretrained models.


In [ ]:

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


In [ ]:

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return correct / total



# Part 1 — ResNet50 Feature Extraction


In [ ]:

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze backbone
for param in resnet.parameters():
    param.requires_grad = False

# Replace classifier
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, num_classes)

resnet = resnet.to(device)

print("Trainable parameters:", count_trainable_params(resnet))


In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.fc.parameters(), lr=0.001)

def train(model, optimizer, epochs=3):
    best_model = copy.deepcopy(model.state_dict())
    best_acc = 0
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        acc = evaluate(model, test_loader)
        print(f"Epoch {epoch+1}, Loss: {running_loss:.3f}, Test Acc: {acc:.4f}")
        
        if acc > best_acc:
            best_acc = acc
            best_model = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_model)
    return model


In [ ]:

resnet = train(resnet, optimizer, epochs=3)



# Part 2 — Fine-Tuning Last Block


In [ ]:

for name, param in resnet.named_parameters():
    if "layer4" in name:
        param.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()),
    lr=1e-4
)

print("Trainable parameters after unfreezing:", count_trainable_params(resnet))

resnet = train(resnet, optimizer, epochs=3)



# Part 3 — MobileNetV3 Comparison


In [ ]:

mobilenet = models.mobilenet_v3_large(
    weights=models.MobileNet_V3_Large_Weights.DEFAULT
)

for param in mobilenet.parameters():
    param.requires_grad = False

num_features = mobilenet.classifier[3].in_features
mobilenet.classifier[3] = nn.Linear(num_features, num_classes)

mobilenet = mobilenet.to(device)

print("Trainable parameters (MobileNet):", count_trainable_params(mobilenet))

optimizer = optim.Adam(mobilenet.classifier.parameters(), lr=0.001)

mobilenet = train(mobilenet, optimizer, epochs=3)



# FINAL TASK (2 Hours – Individual)

## Objective
Demonstrate the impact of domain shift on transfer learning.

## Requirements

1. Select ONLY 3 CIFAR-10 classes.
2. Train using clean images.
3. Corrupt validation set with Gaussian noise or heavy augmentation.
4. Compare:
   - Frozen backbone
   - Fine-tuned backbone
5. Modify ONE architectural element:
   - Add Dropout
   - Add extra FC layer
   - Change optimizer
   - Add weight decay
   - Use label smoothing
6. Measure:
   - Accuracy
   - Training time
   - Average softmax confidence

## Deliverable

3-minute in-class presentation explaining:
- What you modified
- Why
- Results
- One technical insight about transfer learning
